In [1]:
print('Hello World!')

Hello World!


In [4]:
import pandas as pd

inputs = [
    "Why do production RAG systems in 2025 use hybrid search instead of relying only on pure dense vector search?",
    "What is the difference between a child chunk and a parent chunk in advanced ingestion pipelines, and why are they separated?",
    """"How does a two-stage retrieval pipeline address the "lost in the middle" attention problem in LLMs?"""
]

outputs = [
    "Pure dense vector search struggles to find exact keyword matches, serial numbers, product SKUs, and specific alphanumeric codes. Hybrid search combines dense vector embeddings (for conceptual meaning) with sparse token algorithms like BM25 or SPLADE (for exact keywords), merging them via Reciprocal Rank Fusion (RRF) to capture both semantic intent and precise identifiers.",
    "Child chunks are small, granular segments (100–200 tokens) optimized for precise vector search matching. Parent chunks are the larger surrounding context windows (1000–2000 tokens). They are separated to optimize both search accuracy and LLM comprehension, passing the broader context to the LLM so it can understand the overall structure without losing granularity during the initial search phase.",
    "In stage one, a fast vector search narrows millions of documents down to the top 100 candidates. In stage two, a specialized Cross-Encoder Reranker deep-scores and reorders those 100 documents down to the definitive top 5. This moves the most highly relevant context directly to the top of the prompt, preventing the LLM from ignoring critical information buried in the center of long contexts."
]

qa_pairs = [
    {"question": q, "answer": a} for q, a in zip(inputs, outputs)
]

df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = r"D:\LLMOPS\notebook\goldens.csv"
df.to_csv(csv_path, index=False)

In [7]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport"
)

client.create_examples(
    inputs=[{'question': q} for q in inputs],
    outputs=[{'answer': a} for a in outputs],
    dataset_id=dataset.id
)

{'example_ids': ['8f9ff1c8-410b-437f-95b4-e006c80c1ba5',
  '15fae9d4-0fb0-454f-81c7-382001108a2a',
  'c7e17abf-4c97-4929-9374-f344a3c4fd46'],
 'count': 3,
 'as_of': '2026-05-24T10:07:18.964014053Z'}

In [8]:
import sys
sys.path.append(r"D:\LLMOPS")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = r"D:\LLMOPS\data\The_2025_AI_Engineering_Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [9]:
# Test the function with a sample question
test_input = {"question": "Why do production RAG systems in 2025 use hybrid search instead of relying only on pure dense vector search?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-05-24T10:11:00.674727Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-05-24T10:11:00.675385Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-05-24T10:11:00.676044Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"timestamp": "2026-05-24T10:11:00.677039Z", "level": "info", "event": "Loaded HUGGINGFACEHUB_ACCESS_TOKEN from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_yz...", "GOOGLE_API_KEY": "AIzaSy...", "HUGGINGFACEHUB_ACCESS_TOKEN": "hf_mGZ..."}, "timestamp": "2026-05-24T10:11:00.677816Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-24T10:11:00.680575Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260524_154100_f435402a", "temp_dir": "data\\session_20260524_154100_f435402a", "faiss_dir": "faiss_index\\session_20260524_154100_f435402a", 

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Question: Why do production RAG systems in 2025 use hybrid search instead of relying only on pure dense vector search?

Answer: Production RAG systems in 2025 use hybrid search because pure dense vector search struggles with structured or semi‑structured data such as product SKUs, code variables, and user IDs. By combining keyword indexing with vector retrieval from day one, hybrid search improves accuracy and reduces hallucinations for these data types.


In [10]:
from langsmith.evaluation import evaluate, StringEvaluator

In [11]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-05-24T10:13:27.468912Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-05-24T10:13:27.470223Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-05-24T10:13:27.471227Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"timestamp": "2026-05-24T10:13:27.472106Z", "level": "info", "event": "Loaded HUGGINGFACEHUB_ACCESS_TOKEN from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_yz...", "GOOGLE_API_KEY": "AIzaSy...", "HUGGINGFACEHUB_ACCESS_TOKEN": "hf_mGZ..."}, "timestamp": "2026-05-24T10:13:27.472972Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-05-24T10:13:27.477125Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260524_154327_bd5648bb", "temp_dir": "data\\session_20260524_154327_bd5648bb", "faiss_dir": "faiss_index\\session_20260524_154327_bd5648bb", 

Testing all questions from the dataset:

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

...

Q2: What is the difference between a child chunk and a parent chunk in advanced ingestion pipelines, and why are they separated?
A2: Child chunks are small, highly granular pieces (≈100‑200 tokens or single sentences) that are indexed in the vector store for precise, fast retrieval. Parent chunks are larger windows (≈1,000‑2,000 tokens) that surround a matched child chunk and are supplied to the LLM during generation. Separating them lets the system perform efficient, focused retrieval while giving the model a richer context without overwhelming it with unnecessary tokens.

--------------------------------------------------------------------------------

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

DEBUG → embedding model: sentence-transformers/all-MiniLM-L6-v2
DEBUG → using device: cpu


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q3: "How does a two-stage retrieval pipeline address the "lost in the middle" attention problem in LLMs?
A3: A two‑stage retrieval pipeline first uses fast vector search to cut a huge corpus down to a small set (e.g., 100 chunks). Then a cross‑encoder reranker re‑orders those chunks by deep semantic similarity, producing a short, top‑5 list that is placed at the beginning of the prompt. By limiting the prompt length and front‑loading the most relevant context, the LLM no longer has to “look past” middle‑section noise, effectively neutralizing the lost‑in‑the‑middle attention problem.

--------------------------------------------------------------------------------

